In [ ]:
!pip install -q \
  newsapi-python \
  spacy \
  neo4j \
  langchain \
  langchain-groq \
  langchain-community \
  sentence-transformers \
  transformers \
  torch \
  streamlit \
  pyngrok \
  python-dotenv

!python -m spacy download en_core_web_trf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import userdata

NEWS_API_KEY    = userdata.get('NEWS_API_KEY')
NEO4J_URI       = userdata.get('NEO4J_URI')       
NEO4J_USER      = "user-name"
NEO4J_PASSWORD  = userdata.get('NEO4J_PASSWORD')
GROQ_API_KEY    = userdata.get('GROQ_API_KEY')

# In Colab: click the key icon (Secrets) on the left sidebar
# Add each key there — never hardcode API keys in notebooks!

In [ ]:
from newsapi import NewsApiClient
from datetime import datetime, timedelta

newsapi = NewsApiClient(api_key=NEWS_API_KEY)

SUPPLY_CHAIN_QUERIES = [
    "supply chain disruption",
    "semiconductor shortage",
    "port congestion delay",
    "factory shutdown flood earthquake",
    "shipping route blockage",
    "raw material shortage manufacturing"
]

def fetch_articles(query, days_back=7):
    from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    response = newsapi.get_everything(
        q=query,
        from_param=from_date,
        language='en',
        sort_by='relevancy',
        page_size=20
    )
    return response.get('articles', [])

all_articles = []
for query in SUPPLY_CHAIN_QUERIES:
    articles = fetch_articles(query)
    all_articles.extend(articles)
    print(f"Query: '{query}' → {len(articles)} articles")

print(f"\nTotal articles fetched: {len(all_articles)}")

Query: 'supply chain disruption' → 18 articles
Query: 'semiconductor shortage' → 20 articles
Query: 'port congestion delay' → 1 articles
Query: 'factory shutdown flood earthquake' → 0 articles
Query: 'shipping route blockage' → 6 articles
Query: 'raw material shortage manufacturing' → 4 articles

Total articles fetched: 49


In [ ]:
def clean_article(article):
    """Extract only what we need, skip empty content."""
    content = article.get('content') or article.get('description') or ''
    return {
        'title':       article.get('title', ''),
        'text':        article.get('title', '') + '. ' + content,
        'source':      article.get('source', {}).get('name', 'Unknown'),
        'published_at': article.get('publishedAt', ''),
        'url':         article.get('url', '')
    }

cleaned = [clean_article(a) for a in all_articles if a.get('title')]

# Deduplicate by title
seen_titles = set()
unique_articles = []
for art in cleaned:
    if art['title'] not in seen_titles:
        seen_titles.add(art['title'])
        unique_articles.append(art)

print(f"Unique articles after dedup: {len(unique_articles)}")
print("\nSample article text:")
print(unique_articles[0]['text'][:300])

Unique articles after dedup: 45

Sample article text:
UK intelligence censored report on global warming and homeland security. The UK governments latest national security assessment on global biodiversity loss and ecosystem collapse is generating very few public discussions yet its conclusions are alarming. Produced by the U… [+17224 chars]


In [ ]:
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.load("en_core_web_trf")

DISRUPTION_KEYWORDS = [
    "shutdown", "closure", "shortage", "delay", "disruption",
    "blockage", "congestion", "flood", "earthquake", "strike",
    "sanctions", "tariff", "export ban", "factory fire",
    "power outage", "chip shortage", "port closed"
]

# Build a PhraseMatcher for disruption signals
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(kw) for kw in DISRUPTION_KEYWORDS]
matcher.add("DISRUPTION", patterns)

def extract_entities(text):
    """Extract structured entities from one article."""
    doc = nlp(text[:1000])  # limit to 1000 chars for Colab speed

    entities = {
        'companies':   [],
        'locations':   [],
        'events':      [],
        'products':    [],
        'disruptions': []
    }

    for ent in doc.ents:
        if ent.label_ == "ORG":
            entities['companies'].append(ent.text.strip())
        elif ent.label_ in ("GPE", "LOC"):
            entities['locations'].append(ent.text.strip())
        elif ent.label_ == "EVENT":
            entities['events'].append(ent.text.strip())
        elif ent.label_ == "PRODUCT":
            entities['products'].append(ent.text.strip())

    # Catch disruption signal keywords
    matches = matcher(doc)
    for _, start, end in matches:
        entities['disruptions'].append(doc[start:end].text.lower())

    # Deduplicate all lists
    for key in entities:
        entities[key] = list(set(entities[key]))

    return entities

In [ ]:
processed_articles = []

for i, article in enumerate(unique_articles):
    entities = extract_entities(article['text'])
    processed_articles.append({
        **article,
        'entities': entities
    })
    if i % 10 == 0:
        print(f"Processed {i}/{len(unique_articles)} articles...")

# Inspect a result
sample = processed_articles[0]
print(f"\nTitle: {sample['title']}")
print(f"Companies:   {sample['entities']['companies']}")
print(f"Locations:   {sample['entities']['locations']}")
print(f"Disruptions: {sample['entities']['disruptions']}")

Processed 0/45 articles...
Processed 10/45 articles...
Processed 20/45 articles...
Processed 30/45 articles...
Processed 40/45 articles...

Title: UK intelligence censored report on global warming and homeland security
Companies:   ['U']
Locations:   ['UK']
Disruptions: []


In [ ]:
from collections import Counter

all_companies   = []
all_locations   = []
all_disruptions = []

for art in processed_articles:
    all_companies.extend(art['entities']['companies'])
    all_locations.extend(art['entities']['locations'])
    all_disruptions.extend(art['entities']['disruptions'])

print("Top 15 companies in news:")
for name, count in Counter(all_companies).most_common(15):
    print(f"  {count:3d}x  {name}")

print("\nTop 10 disruption signals:")
for name, count in Counter(all_disruptions).most_common(10):
    print(f"  {count:3d}x  {name}")

Top 15 companies in news:
    5x  Samsung Electronics
    3x  Samsung
    1x  U
    1x  Amazon
    1x  Axios
    1x  →Disney
    1x  Disney
    1x  Mediaocean
    1x  Digiday+
    1x  IFDA
    1x  Iran Press TV
    1x  Food and Drug Administration
    1x  CERT-EU
    1x  European Commission
    1x  TeamPCP

Top 10 disruption signals:
    4x  shortage
    3x  disruption
    1x  strike
    1x  closure


In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

# Test connection
result = run_query("RETURN 'Connected to Neo4j!' AS msg")
print(result[0]['msg'])

Connected to Neo4j!


In [ ]:
CONSTRAINTS = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company)  REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Country)  REQUIRE l.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (e:Event)    REQUIRE e.id   IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Product)  REQUIRE p.name IS UNIQUE",
]

for constraint in CONSTRAINTS:
    run_query(constraint)
    print(f"Created: {constraint[:60]}...")

Created: CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company)  REQUIRE c.n...
Created: CREATE CONSTRAINT IF NOT EXISTS FOR (l:Country)  REQUIRE l.n...
Created: CREATE CONSTRAINT IF NOT EXISTS FOR (e:Event)    REQUIRE e.i...
Created: CREATE CONSTRAINT IF NOT EXISTS FOR (p:Product)  REQUIRE p.n...


In [ ]:
import hashlib

def ingest_article_to_graph(article):
    entities  = article['entities']
    companies = entities['companies'][:5]   # cap to avoid noise
    locations = entities['locations'][:5]
    disrupts  = entities['disruptions'][:3]

    event_id = hashlib.md5(article['url'].encode()).hexdigest()[:12]

    with driver.session() as session:
        # 1. Create an Event node for this article
        session.run("""
            MERGE (e:Event {id: $event_id})
            SET e.title       = $title,
                e.source      = $source,
                e.published_at = $published_at,
                e.url          = $url,
                e.disruption_signals = $disrupts
        """, event_id=event_id, title=article['title'],
             source=article['source'],
             published_at=article['published_at'],
             url=article['url'], disrupts=disrupts)

        # 2. Create Company nodes + link to event
        for company in companies:
            session.run("""
                MERGE (c:Company {name: $name})
                WITH c
                MATCH (e:Event {id: $event_id})
                MERGE (c)-[:MENTIONED_IN]->(e)
            """, name=company, event_id=event_id)

        # 3. Create Country nodes + link companies to countries
        for location in locations:
            session.run("""
                MERGE (co:Country {name: $name})
                WITH co
                MATCH (e:Event {id: $event_id})
                MERGE (e)-[:LOCATED_IN]->(co)
            """, name=location, event_id=event_id)

        # 4. Link companies to locations (co-occurrence = possible supplier relationship)
        for company in companies:
            for location in locations:
                session.run("""
                    MATCH (c:Company {name: $company})
                    MATCH (co:Country {name: $location})
                    MERGE (c)-[:OPERATES_IN]->(co)
                """, company=company, location=location)

# Run ingestion
for i, article in enumerate(processed_articles):
    ingest_article_to_graph(article)

print("Graph ingestion complete!")
result = run_query("MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count")
for row in result:
    print(f"  {row['type']:15s}: {row['count']} nodes")

Graph ingestion complete!
  Event          : 83 nodes
  Company        : 84 nodes
  Country        : 56 nodes
  Product        : 7 nodes


In [ ]:
# Seed with known real-world supply chain facts
# This dramatically improves multi-hop reasoning quality

KNOWN_SUPPLY_CHAINS = [
    ("Apple",   "TSMC",        "semiconductors"),
    ("Apple",   "Foxconn",     "assembly"),
    ("NVIDIA",  "TSMC",        "GPUs"),
    ("Samsung", "ASML",        "lithography"),
    ("Toyota",  "Denso",       "auto parts"),
    ("Tesla",   "Panasonic",   "batteries"),
    ("Boeing",  "Spirit AeroSystems", "fuselage"),
]

KNOWN_LOCATIONS = [
    ("TSMC",     "Taiwan"),
    ("Foxconn",  "China"),
    ("Foxconn",  "India"),
    ("Samsung",  "South Korea"),
    ("Toyota",   "Japan"),
    ("Tesla",    "United States"),
    ("Panasonic","Japan"),
]

with driver.session() as session:
    for buyer, supplier, product in KNOWN_SUPPLY_CHAINS:
        session.run("""
            MERGE (b:Company {name: $buyer})
            MERGE (s:Company {name: $supplier})
            MERGE (p:Product  {name: $product})
            MERGE (s)-[:SUPPLIES_TO {product: $product}]->(b)
            MERGE (s)-[:PRODUCES]->(p)
        """, buyer=buyer, supplier=supplier, product=product)

    for company, country in KNOWN_LOCATIONS:
        session.run("""
            MERGE (c:Company {name: $company})
            MERGE (co:Country {name: $country})
            MERGE (c)-[:HEADQUARTERED_IN]->(co)
        """, company=company, country=country)

print("Seed data loaded. Graph is ready for querying!")

Seed data loaded. Graph is ready for querying!


In [ ]:
def get_company_subgraph(company_name, depth=2):
    """
    Retrieve a subgraph around a company:
    - Its direct suppliers
    - Countries those suppliers operate in
    - Recent disruption events in those countries
    - Risk signals attached to events
    """
    result = run_query("""
        MATCH (target:Company {name: $name})

        OPTIONAL MATCH (supplier:Company)-[:SUPPLIES_TO]->(target)
        OPTIONAL MATCH (supplier)-[:HEADQUARTERED_IN|OPERATES_IN]->(country:Country)
        OPTIONAL MATCH (event:Event)-[:LOCATED_IN]->(country)

        RETURN
            target.name                          AS company,
            collect(DISTINCT supplier.name)      AS suppliers,
            collect(DISTINCT country.name)       AS countries,
            collect(DISTINCT event.title)[..5]   AS recent_events,
            collect(DISTINCT event.disruption_signals)[..5] AS risk_signals
    """, params={'name': company_name})

    return result[0] if result else None

# Test it
subgraph = get_company_subgraph("Apple")
if subgraph:
    print(f"Company:      {subgraph['company']}")
    print(f"Suppliers:    {subgraph['suppliers']}")
    print(f"Countries:    {subgraph['countries']}")
    print(f"Events:       {subgraph['recent_events'][:3]}")
    print(f"Risk signals: {subgraph['risk_signals'][:3]}")

Company:      Apple
Suppliers:    ['TSMC', 'Foxconn']
Countries:    ['China', 'Arizona', 'Taiwan', 'India']
Events:       ["Weekly news roundup: China's special AI chip supply ends; TSMC plans 12 fabs in Arizona", 'Chinese chip firms hit record high revenue driven by the AI boom and U.S. curbs', 'China warns of fresh chip shortage as Nexperia dispute escalates again - Dutch headquarters allegedly locked Chinese staff out of IT systems']
Risk signals: [[], ['shortage'], ['chip shortage', 'shortage']]


In [ ]:
def subgraph_to_context(subgraph):
    """
    Convert the structured subgraph dict into a
    plain-English context string for the LLM prompt.
    This is the 'retrieval' output in Graph RAG.
    """
    if not subgraph:
        return "No graph data found for this company."

    lines = []
    lines.append(f"COMPANY: {subgraph['company']}")

    if subgraph['suppliers']:
        lines.append(f"KNOWN SUPPLIERS: {', '.join(subgraph['suppliers'])}")

    if subgraph['countries']:
        lines.append(f"SUPPLIER COUNTRIES: {', '.join(subgraph['countries'])}")

    if subgraph['recent_events']:
        lines.append("RECENT NEWS EVENTS:")
        for evt in subgraph['recent_events'][:5]:
            lines.append(f"  - {evt}")

    if subgraph['risk_signals']:
        flat_signals = []
        for sig_list in subgraph['risk_signals']:
            if isinstance(sig_list, list):
                flat_signals.extend(sig_list)
            else:
                flat_signals.append(sig_list)
        if flat_signals:
            lines.append(f"DETECTED RISK SIGNALS: {', '.join(set(flat_signals))}")

    return '\n'.join(lines)

context = subgraph_to_context(get_company_subgraph("Apple"))
print(context)

COMPANY: Apple
KNOWN SUPPLIERS: TSMC, Foxconn
SUPPLIER COUNTRIES: China, Arizona, Taiwan, India
RECENT NEWS EVENTS:
  - Weekly news roundup: China's special AI chip supply ends; TSMC plans 12 fabs in Arizona
  - Chinese chip firms hit record high revenue driven by the AI boom and U.S. curbs
  - China warns of fresh chip shortage as Nexperia dispute escalates again - Dutch headquarters allegedly locked Chinese staff out of IT systems
  - Himax Technologies, Inc. Schedules First Quarter 2026 Financial Results Conference Call on Thursday, May 7, 2026 at 8:00 AM EDT
  - Why MRI scans In India could get costlier — and slower — because of the West Asia war
DETECTED RISK SIGNALS: chip shortage, shortage


In [ ]:
def find_risk_paths(company_name):
    """
    Multi-hop: Company → Supplier → Country → Event
    Returns all risk paths as readable strings.
    """
    paths = run_query("""
        MATCH path = (c:Company {name: $name})
            <-[:SUPPLIES_TO]-(supplier:Company)
            -[:HEADQUARTERED_IN|OPERATES_IN]->(country:Country)
            <-[:LOCATED_IN]-(event:Event)
        WHERE size(event.disruption_signals) > 0
        RETURN
            supplier.name       AS supplier,
            country.name        AS country,
            event.title         AS event_title,
            event.disruption_signals AS signals
        LIMIT 10
    """, params={'name': company_name})

    return paths

paths = find_risk_paths("Apple")
print(f"Risk paths found: {len(paths)}")
for p in paths:
    print(f"\n  {p['supplier']} (in {p['country']})")
    print(f"  Event: {p['event_title'][:80]}")
    print(f"  Signals: {p['signals']}")

Risk paths found: 4

  TSMC (in China)
  Event: Chinese chip firms hit record high revenue driven by the AI boom and U.S. curbs
  Signals: ['shortage']

  TSMC (in China)
  Event: China warns of fresh chip shortage as Nexperia dispute escalates again - Dutch h
  Signals: ['chip shortage', 'shortage']

  Foxconn (in China)
  Event: Chinese chip firms hit record high revenue driven by the AI boom and U.S. curbs
  Signals: ['shortage']

  Foxconn (in China)
  Event: China warns of fresh chip shortage as Nexperia dispute escalates again - Dutch h
  Signals: ['chip shortage', 'shortage']


In [ ]:

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0.1       # low temp = consistent risk reports
)

SYSTEM_PROMPT = """You are a supply chain risk analyst.
You will be given a knowledge graph context about a company, its suppliers,
the countries they operate in, and recent news events + risk signals.

Your job is to:
1. Identify the most critical supply chain risks (1-5 scale)
2. Trace WHY each risk exists using the supplier → country → event chain
3. Suggest mitigation strategies
4. Give an overall risk score (1-10) with justification

Be specific. Reference actual company names and events from the context.
If data is limited, say so — do not hallucinate relationships."""

In [ ]:
def assess_supply_chain_risk(company_name):
    """
    Full Graph RAG + LLM pipeline:
    1. Query graph → get subgraph context
    2. Find multi-hop risk paths
    3. Serialise context
    4. Call LLM for structured risk assessment
    """
    print(f"Analyzing supply chain risk for: {company_name}")
    print("-" * 50)

    # Step 1: Retrieve subgraph context
    subgraph  = get_company_subgraph(company_name)
    context   = subgraph_to_context(subgraph)

    # Step 2: Get multi-hop risk paths
    risk_paths = find_risk_paths(company_name)

    path_text = ""
    if risk_paths:
        path_text = "\nRISK CHAIN PATHS (multi-hop):\n"
        for p in risk_paths[:5]:
            path_text += (
                f"  {company_name} ← {p['supplier']} "
                f"(located in {p['country']}) "
                f"← [{p['event_title'][:60]}] "
                f"(signals: {p['signals']})\n"
            )

    full_context = context + path_text

    # Step 3: LLM reasoning
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"""
Analyse the supply chain disruption risk for {company_name}.

KNOWLEDGE GRAPH CONTEXT:
{full_context}

Provide:
- Overall risk score (1-10)
- Top 3 specific risks with evidence from the graph
- The most critical supplier-country-event chain
- 3 actionable mitigation recommendations
""")
    ]

    response = llm.invoke(messages)
    return response.content

# Run the full pipeline
report = assess_supply_chain_risk("Apple")
print(report)

Analyzing supply chain risk for: Apple
--------------------------------------------------
**Overall Risk Score: 8/10**

Apple's supply chain faces significant risks due to the complex interplay of factors in the global semiconductor industry. The recent news events and detected risk signals indicate a high likelihood of disruptions in the supply chain.

**Top 3 Specific Risks:**

1. **Risk: Chip shortage due to TSMC's reliance on Chinese suppliers**
Evidence: Apple ← TSMC (located in China) ← [Chinese chip firms hit record high revenue driven by the AI ] (signals: ['shortage'])
TSMC's dependence on Chinese suppliers increases the risk of a chip shortage, which could impact Apple's production.
2. **Risk: Chip shortage due to Nexperia dispute and China's warning**
Evidence: Apple ← TSMC (located in China) ← [China warns of fresh chip shortage as Nexperia dispute escal] (signals: ['chip shortage', 'shortage'])
The ongoing dispute between Nexperia and the Chinese government, combined with 

In [ ]:
import re

def extract_risk_score(llm_output):
    """Parse the numerical risk score from the LLM's text output."""
    patterns = [
        r'[Oo]verall [Rr]isk [Ss]core[:\s]+(\d+(?:\.\d+)?)',
        r'[Rr]isk [Ss]core[:\s]+(\d+(?:\.\d+)?)/10',
        r'\*\*(\d+(?:\.\d+)?)/10\*\*',
        r'Score:\s*(\d+(?:\.\d+)?)',
    ]
    for pattern in patterns:
        match = re.search(pattern, llm_output)
        if match:
            return float(match.group(1))
    return None   # could not parse

score = extract_risk_score(report)
print(f"Extracted risk score: {score}/10")

Extracted risk score: 8.0/10


In [ ]:
%%writefile app.py
import streamlit as st
import os
import re
import hashlib
from collections import Counter

from newsapi import NewsApiClient
from neo4j import GraphDatabase
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import spacy
from spacy.matcher import PhraseMatcher

# ── Secrets ───────────────────────────────────────────────────────────────────
NEWS_API_KEY   = os.environ.get("NEWS_API_KEY", "")
NEO4J_URI      = os.environ.get("NEO4J_URI", "")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")
GROQ_API_KEY   = os.environ.get("GROQ_API_KEY", "")
NEO4J_USER     = "eb10e102"

# ── Neo4j ─────────────────────────────────────────────────────────────────────
@st.cache_resource
def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query, params=None):
    with get_driver().session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

# ── spaCy ─────────────────────────────────────────────────────────────────────
@st.cache_resource
def load_nlp():
    nlp = spacy.load("en_core_web_trf")
    DISRUPTION_KEYWORDS = [
        "shutdown", "closure", "shortage", "delay", "disruption",
        "blockage", "congestion", "flood", "earthquake", "strike",
        "sanctions", "tariff", "export ban", "factory fire",
        "power outage", "chip shortage", "port closed"
    ]
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns = [nlp.make_doc(kw) for kw in DISRUPTION_KEYWORDS]
    matcher.add("DISRUPTION", patterns)
    return nlp, matcher

# ── NER ───────────────────────────────────────────────────────────────────────
def extract_entities(text):
    nlp, matcher = load_nlp()
    doc = nlp(text[:1000])
    entities = {"companies": [], "locations": [], "events": [], "disruptions": []}
    for ent in doc.ents:
        if ent.label_ == "ORG":
            entities["companies"].append(ent.text.strip())
        elif ent.label_ in ("GPE", "LOC"):
            entities["locations"].append(ent.text.strip())
        elif ent.label_ == "EVENT":
            entities["events"].append(ent.text.strip())
    matches = matcher(doc)
    for _, start, end in matches:
        entities["disruptions"].append(doc[start:end].text.lower())
    for key in entities:
        entities[key] = list(set(entities[key]))
    return entities

# ── News ingestion ────────────────────────────────────────────────────────────
def fetch_and_ingest_news():
    newsapi = NewsApiClient(api_key=NEWS_API_KEY)
    queries = [
        "supply chain disruption", "semiconductor shortage",
        "port congestion delay", "factory shutdown flood earthquake",
    ]
    all_articles = []
    for q in queries:
        try:
            resp = newsapi.get_everything(q=q, language="en",
                                          sort_by="relevancy", page_size=15)
            all_articles.extend(resp.get("articles", []))
        except Exception:
            pass

    seen, unique = set(), []
    for a in all_articles:
        if a.get("title") and a["title"] not in seen:
            seen.add(a["title"])
            content = a.get("content") or a.get("description") or ""
            unique.append({
                "title": a["title"],
                "text":  a["title"] + ". " + content,
                "source": a.get("source", {}).get("name", "Unknown"),
                "published_at": a.get("publishedAt", ""),
                "url": a.get("url", ""),
            })

    # Ingest into Neo4j
    constraints = [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Country) REQUIRE l.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (e:Event)   REQUIRE e.id   IS UNIQUE",
    ]
    for c in constraints:
        run_query(c)

    for article in unique:
        entities = extract_entities(article["text"])
        event_id = hashlib.md5(article["url"].encode()).hexdigest()[:12]
        with get_driver().session() as session:
            session.run("""
                MERGE (e:Event {id: $event_id})
                SET e.title=$title, e.source=$source,
                    e.published_at=$published_at, e.url=$url,
                    e.disruption_signals=$disrupts
            """, event_id=event_id, title=article["title"],
                 source=article["source"], published_at=article["published_at"],
                 url=article["url"], disrupts=entities["disruptions"])
            for company in entities["companies"][:5]:
                session.run("""
                    MERGE (c:Company {name:$name})
                    WITH c MATCH (e:Event {id:$eid})
                    MERGE (c)-[:MENTIONED_IN]->(e)
                """, name=company, eid=event_id)
            for loc in entities["locations"][:5]:
                session.run("""
                    MERGE (co:Country {name:$name})
                    WITH co MATCH (e:Event {id:$eid})
                    MERGE (e)-[:LOCATED_IN]->(co)
                """, name=loc, eid=event_id)
            for company in entities["companies"][:5]:
                for loc in entities["locations"][:5]:
                    session.run("""
                        MATCH (c:Company {name:$company})
                        MATCH (co:Country {name:$loc})
                        MERGE (c)-[:OPERATES_IN]->(co)
                    """, company=company, loc=loc)

    # Seed known supply chains
    known_chains = [
        ("Apple","TSMC","semiconductors"), ("Apple","Foxconn","assembly"),
        ("NVIDIA","TSMC","GPUs"), ("Samsung","ASML","lithography"),
        ("Toyota","Denso","auto parts"), ("Tesla","Panasonic","batteries"),
        ("Boeing","Spirit AeroSystems","fuselage"),
    ]
    known_locs = [
        ("TSMC","Taiwan"), ("Foxconn","China"), ("Foxconn","India"),
        ("Samsung","South Korea"), ("Toyota","Japan"),
        ("Tesla","United States"), ("Panasonic","Japan"),
    ]
    with get_driver().session() as session:
        for buyer, supplier, product in known_chains:
            session.run("""
                MERGE (b:Company {name:$buyer})
                MERGE (s:Company {name:$supplier})
                MERGE (s)-[:SUPPLIES_TO {product:$product}]->(b)
            """, buyer=buyer, supplier=supplier, product=product)
        for company, country in known_locs:
            session.run("""
                MERGE (c:Company {name:$company})
                MERGE (co:Country {name:$country})
                MERGE (c)-[:HEADQUARTERED_IN]->(co)
            """, company=company, country=country)

    return len(unique)

# ── Graph queries ─────────────────────────────────────────────────────────────
def get_company_subgraph(company_name, depth=2):
    """
    Retrieve a subgraph around a company:
    - Its direct suppliers
    - Countries those suppliers operate in
    - Recent disruption events in those countries
    - Risk signals attached to events
    """
    result = run_query("""
        MATCH (target:Company {name: $name})

        OPTIONAL MATCH (supplier:Company)-[:SUPPLIES_TO]->(target)
        OPTIONAL MATCH (supplier)-[:HEADQUARTERED_IN|OPERATES_IN]->(country:Country)
        OPTIONAL MATCH (event:Event)-[:LOCATED_IN]->(country)

        RETURN
            target.name                          AS company,
            collect(DISTINCT supplier.name)      AS suppliers,
            collect(DISTINCT country.name)       AS countries,
            collect(DISTINCT event.title)[..5]   AS recent_events,
            collect(DISTINCT event.disruption_signals)[..5] AS risk_signals
    """, params={'name': company_name})

    return result[0] if result else None

def subgraph_to_context(subgraph):
    if not subgraph:
        return "No graph data found for this company."
    lines = [f"COMPANY: {subgraph['company']}"]
    if subgraph["suppliers"]:
        lines.append(f"KNOWN SUPPLIERS: {', '.join(subgraph['suppliers'])}")
    if subgraph["countries"]:
        lines.append(f"SUPPLIER COUNTRIES: {', '.join(subgraph['countries'])}")
    if subgraph["recent_events"]:
        lines.append("RECENT NEWS EVENTS:")
        for evt in subgraph["recent_events"][:5]:
            if evt:
                lines.append(f"  - {evt}")
    flat_signals = []
    for sig in subgraph.get("risk_signals", []):
        if isinstance(sig, list):
            flat_signals.extend(sig)
        elif sig:
            flat_signals.append(sig)
    if flat_signals:
        lines.append(f"DETECTED RISK SIGNALS: {', '.join(set(flat_signals))}")
    return "\n".join(lines)

def find_risk_paths(company_name):
    """
    Multi-hop: Company → Supplier → Country → Event
    Returns all risk paths as readable strings.
    """
    paths = run_query("""
        MATCH path = (c:Company {name: $name})
            <-[:SUPPLIES_TO]-(supplier:Company)
            -[:HEADQUARTERED_IN|OPERATES_IN]->(country:Country)
            <-[:LOCATED_IN]-(event:Event)
        WHERE size(event.disruption_signals) > 0
        RETURN
            supplier.name       AS supplier,
            country.name        AS country,
            event.title         AS event_title,
            event.disruption_signals AS signals
        LIMIT 10
    """, params={'name': company_name})

    return paths

# ── LLM ───────────────────────────────────────────────────────────────────────
@st.cache_resource
def get_llm():
    return ChatGroq(groq_api_key=GROQ_API_KEY,
                    model_name="llama-3.1-8b-instant", temperature=0.1)

SYSTEM_PROMPT = """You are a supply chain risk analyst.
Given a knowledge graph context about a company, its suppliers, their countries,
and recent news events with risk signals, you must:
1. Give an overall risk score (1-10)
2. List the top 3 specific risks with evidence
3. Identify the most critical supplier → country → event chain
4. Give 3 actionable mitigation recommendations
Be specific. Reference actual names from the context. Do not hallucinate."""

def assess_supply_chain_risk(company_name):
    subgraph   = get_company_subgraph(company_name)
    context    = subgraph_to_context(subgraph)
    risk_paths = find_risk_paths(company_name)
    path_text  = ""
    if risk_paths:
        path_text = "\nRISK CHAIN PATHS:\n"
        for p in risk_paths[:5]:
            path_text += (f"  {company_name} ← {p['supplier']} "
                          f"(in {p['country']}) ← "
                          f"[{p['event_title'][:60]}] "
                          f"signals: {p['signals']}\n")
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Analyse risk for {company_name}.\n\n"
                              f"CONTEXT:\n{context}{path_text}")
    ]
    return get_llm().invoke(messages).content

def extract_risk_score(text):
    for pattern in [
        r'[Oo]verall [Rr]isk [Ss]core[:\s]+(\d+(?:\.\d+)?)',
        r'[Rr]isk [Ss]core[:\s]+(\d+(?:\.\d+)?)/10',
        r'\*\*(\d+(?:\.\d+)?)/10\*\*',
        r'Score:\s*(\d+(?:\.\d+)?)',
    ]:
        m = re.search(pattern, text)
        if m:
            return float(m.group(1))
    return None

# ── Streamlit UI ──────────────────────────────────────────────────────────────
st.set_page_config(page_title="Supply Chain Risk Predictor", layout="wide")
st.title("Supply Chain Disruption Predictor")
st.caption("Knowledge Graph + Graph RAG + Llama 3")

with st.sidebar:
    st.subheader("Data pipeline")
    if st.button("Fetch & ingest news", use_container_width=True):
        with st.spinner("Fetching news and building graph..."):
            n = fetch_and_ingest_news()
        st.success(f"Ingested {n} articles into Neo4j")

    st.divider()
    st.subheader("Analyse a company")
    company = st.text_input("Company name", placeholder="e.g. Apple, Toyota")
    for preset in ["Apple", "Toyota", "NVIDIA", "Boeing", "Tesla"]:
        if st.button(preset, use_container_width=True):
            company = preset
            st.session_state["company"] = preset
    run_btn = st.button("Run risk assessment", type="primary",
                         use_container_width=True)

company = st.session_state.get("company", company)

if run_btn and company:
    col1, col2 = st.columns(2)
    with col1:
        with st.spinner("Querying knowledge graph..."):
            subgraph = get_company_subgraph(company)
            context  = subgraph_to_context(subgraph)
        st.subheader("Graph context")
        st.code(context, language="text")
        paths = find_risk_paths(company)
        if paths:
            st.subheader("Multi-hop risk paths")
            for p in paths[:5]:
                st.markdown(f"**{company}** ← **{p['supplier']}**"
                            f" (in {p['country']}) ← _{p['event_title'][:70]}_")
    with col2:
        with st.spinner("LLM generating risk report..."):
            report = assess_supply_chain_risk(company)
            score  = extract_risk_score(report)
        if score is not None:
            color = "red" if score >= 7 else "orange" if score >= 4 else "green"
            st.metric("Risk score", f"{score:.1f} / 10",
                      delta="High" if score >= 7 else
                            "Medium" if score >= 4 else "Low")
        st.subheader("Full risk assessment")
        st.markdown(report)
elif not company:
    st.info("Run 'Fetch & ingest news' first, then enter a company name in the sidebar.")

Overwriting app.py


In [ ]:
import subprocess, time
from pyngrok import ngrok
from google.colab import userdata # Import userdata to access secrets

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Kill any outstanding ngrok tunnels to free up resources
ngrok.kill()

# Set ngrok authtoken from Colab secrets
# You need to add 'NGROK_AUTH_TOKEN' to your Colab secrets
# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Set environment variables for the Streamlit subprocess
import os
os.environ["NEWS_API_KEY"]   = NEWS_API_KEY
os.environ["NEO4J_URI"]      = NEO4J_URI
os.environ["NEO4J_USER"]     = NEO4J_USER # Add NEO4J_USER to environment variables
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY

subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)
public_url = ngrok.connect(8501)
print(f"Live at: {public_url}")

Live at: NgrokTunnel: "https://unsuffusive-nellie-unaccepted.ngrok-free.dev" -> "http://localhost:8501"
